# Postprocessing Rules Test Notebook
Load a raw transcription CSV and step through each output rule to see its effect.

In [1]:
import sys
import pandas as pd
sys.path.insert(0, '/home1/zrentala/automated_annotation')
from postprocessing import (
    build_context, build_output_rules, apply_output_rules,
    PluralNormalization, ListWordPreference, MultiWordMerge,
    WordpoolFilter, LongDurationVocalization, OnsetAdjust,
    OUTPUT_RULE_REGISTRY,
)

## 1. Configuration
Set the paths here. Change these to test different sessions/trials.

In [2]:
# --- CONFIGURE THESE ---
EXPERIMENT = 'VCBehOnly'
SUBJECT = 'LTP617'
SESSION = 3
TRIAL = 0  # which .wav/.csv file number
TAG = 'whisperx'
SPLIT = 'val'

# Derived paths
session_dir = f'/data/eeg/scalp/ltp/{EXPERIMENT}/{SUBJECT}/session_{SESSION}'
# csv_path = f'/home1/zrentala/automated_annotation/results/{TAG}/{SPLIT}{session_dir}/whisperx_out/{TRIAL}.csv'
csv_path = f'/data/eeg/scalp/ltp/{EXPERIMENT}/{SUBJECT}/session_{SESSION}/{TAG}_{TRIAL}.csv'
lst_path = f'{session_dir}/{TRIAL}.lst'

print(f'CSV:       {csv_path}')
print(f'LST:       {lst_path}')
print(f'Session:   {session_dir}')

CSV:       /data/eeg/scalp/ltp/VCBehOnly/LTP617/session_3/whisperx_0.csv
LST:       /data/eeg/scalp/ltp/VCBehOnly/LTP617/session_3/0.lst
Session:   /data/eeg/scalp/ltp/VCBehOnly/LTP617/session_3


## 2. Load Data & Context

In [3]:
# Load raw transcription
df_raw = pd.read_csv(csv_path)
print(f'Raw transcription: {len(df_raw)} words')
df_raw

Raw transcription: 40 words


,Word,Onset,Offset,Probability
0,DRYER,335,1517,0.726
1,MACHINE,1577,2118,0.960
2,HELMET,2739,3280,0.962
3,PICKLE,4963,5463,0.907
4,EGGS,6345,8148,0.942
5,BAGELS,8649,9330,0.873
6,OREGANO,10251,10972,0.830
7,WRENCH,12134,12615,0.921
8,FISHBOWL,14358,14999,0.865
9,MOUTHWASH,16061,16842,0.882


In [4]:
# Build context (wordpool, list_words, etc.)
context = build_context(session_dir)

# Load per-file list words
import os
if os.path.exists(lst_path):
    with open(lst_path) as f:
        context['file_list_words'] = {line.strip().upper() for line in f if line.strip()}
else:
    context['file_list_words'] = None

print(f"Experiment:       {context['experiment']}")
print(f"Subject:          {context['subject']}")
print(f"Session:          {context['session']}")
print(f"Wordpool size:    {len(context['wordpool']) if context['wordpool'] else 0}")
print(f"All list words:   {len(context['list_words']) if context['list_words'] else 0}")
print(f"This trial's lst: {sorted(context['file_list_words']) if context['file_list_words'] else 'N/A'}")

Experiment:       VCBehOnly
Subject:          LTP617
Session:          3
Wordpool size:    1873
All list words:   150
This trial's lst: ['BAGELS', 'CHALK', 'DRYER MACHINE', 'EGGS', 'FERTILIZER', 'FISH_BOWL', 'GLASSES', 'HELMET', 'JACKET', 'MOUTHWASH', 'NEWSPAPER', 'OREGANO', 'PICKLES', 'VITAMINS', 'WRENCH']


## 3. Step Through Each Rule
Run each rule individually to see what it changes.

In [5]:
def show_diff(before, after, rule_name):
    """Show what changed between before and after DataFrames."""
    print(f'\n{"=" * 60}')
    print(f'  {rule_name}')
    print(f'{"=" * 60}')
    print(f'Rows: {len(before)} -> {len(after)}')
    
    if len(before) == len(after):
        # Compare column-by-column for same-length DFs
        changes = []
        for i in range(len(before)):
            for col in before.columns:
                b = before.iloc[i][col]
                a = after.iloc[i][col]
                if str(b) != str(a):
                    changes.append({'Row': i, 'Column': col, 'Before': b, 'After': a})
        if changes:
            print(pd.DataFrame(changes).to_string(index=False))
        else:
            print('No changes.')
    else:
        print('Row count changed — showing full output:')
        display(after)
    
    return after

In [6]:
df_raw

,Word,Onset,Offset,Probability
0,DRYER,335,1517,0.726
1,MACHINE,1577,2118,0.960
2,HELMET,2739,3280,0.962
3,PICKLE,4963,5463,0.907
4,EGGS,6345,8148,0.942
5,BAGELS,8649,9330,0.873
6,OREGANO,10251,10972,0.830
7,WRENCH,12134,12615,0.921
8,FISHBOWL,14358,14999,0.865
9,MOUTHWASH,16061,16842,0.882


In [7]:
from postprocessing import (
    build_context, build_output_rules, apply_output_rules,
    Normalization, ListWordPreference, MultiWordMerge,
    WordpoolFilter, LongDurationVocalization, OnsetAdjust,
    OUTPUT_RULE_REGISTRY,
)
# Run each rule one at a time, showing diffs
ALL_RULES = [
    ('Normalization', Normalization()),
    ('MultiWordMerge', MultiWordMerge()),
    ('ListWordPreference', ListWordPreference()),
    ('WordpoolFilter', WordpoolFilter()),
    ('LongDurationVocalization', LongDurationVocalization()),
    ('OnsetAdjust', OnsetAdjust()),
]

df_current = df_raw.copy()
for name, rule in ALL_RULES:
    df_after = rule.apply(df_current, context)
    df_current = show_diff(df_current, df_after, name)
df_current


  Normalization
Rows: 40 -> 40
No changes.
['DRYER', 'MACHINE']
['DRYER', 'MACHINE']

  MultiWordMerge
Rows: 40 -> 38
Row count changed — showing full output:


,Word,Onset,Offset,Probability
0,DRYER MACHINE,335,2118,0.726
1,HELMET,2739,3280,0.962
2,PICKLE,4963,5463,0.907
3,EGGS,6345,8148,0.942
4,BAGELS,8649,9330,0.873
5,OREGANO,10251,10972,0.830
6,WRENCH,12134,12615,0.921
7,FISHBOWL,14358,14999,0.865
8,MOUTHWASH,16061,16842,0.882
9,VITAMINS,18264,23473,0.945



  ListWordPreference
Rows: 38 -> 38
 Row      Column  Before  After
   0 Probability   0.726  0.776
   1 Probability   0.962  1.000
   3 Probability   0.942  0.992
   4 Probability   0.873  0.923
   5 Probability   0.830  0.880
   6 Probability   0.921  0.971
   8 Probability   0.882  0.932
   9 Probability   0.945  0.995
  32 Probability   0.635  0.685

  WordpoolFilter
Rows: 38 -> 38
 Row Column    Before After
   2   Word    PICKLE    <>
   7   Word  FISHBOWL    <>
  10   Word      DONT    <>
  11   Word  REMEMBER    <>
  12   Word       THE    <>
  13   Word       GYM    <>
  14   Word       ONE    <>
  15   Word        OR    <>
  16   Word       THE    <>
  17   Word     PARTY    <>
  18   Word     STORE    <>
  19   Word       ONE    <>
  20   Word        OR    <>
  21   Word       THE    <>
  22   Word BOOKSTORE    <>
  23   Word       ONE    <>
  24   Word       BUT    <>
  25   Word     THATS    <>
  26   Word      MOST    <>
  27   Word        OF    <>
  28   Word      THEM 

,Word,Onset,Offset,Probability
0,DRYER MACHINE,335,2118,0.776
1,<>,1335,2118,0.776
2,HELMET,2739,3280,1.000
3,<>,4963,5463,0.907
4,EGGS,6345,8148,0.992
5,<>,7345,8148,0.992
6,BAGELS,8649,9330,0.923
7,OREGANO,10251,10972,0.880
8,WRENCH,12134,12615,0.971
9,<>,14358,14999,0.865



  OnsetAdjust
Rows: 51 -> 51
 Row Column  Before  After
   0  Onset     335    330
   1  Onset    1335   1330
   2  Onset    2739   2734
   3  Onset    4963   4958
   4  Onset    6345   6340
   5  Onset    7345   7340
   6  Onset    8649   8644
   7  Onset   10251  10246
   8  Onset   12134  12129
   9  Onset   14358  14353
  10  Onset   16061  16056
  11  Onset   18264  18259
  12  Onset   19264  19259
  13  Onset   20264  20259
  14  Onset   21264  21259
  15  Onset   22264  22259
  16  Onset   23264  23259
  17  Onset   37308  37303
  18  Onset   37568  37563
  19  Onset   37889  37884
  20  Onset   38049  38044
  21  Onset   38570  38565
  22  Onset   39130  39125
  23  Onset   39250  39245
  24  Onset   39431  39426
  25  Onset   39711  39706
  26  Onset   40192  40187
  27  Onset   40752  40747
  28  Onset   41113  41108
  29  Onset   41373  41368
  30  Onset   42314  42309
  31  Onset   43314  43309
  32  Onset   44314  44309
  33  Onset   45314  45309
  34  Onset   46314  4630

,Word,Onset,Offset,Probability
0,DRYER MACHINE,330,2118,0.776
1,<>,1330,2118,0.776
2,HELMET,2734,3280,1.000
3,<>,4958,5463,0.907
4,EGGS,6340,8148,0.992
5,<>,7340,8148,0.992
6,BAGELS,8644,9330,0.923
7,OREGANO,10246,10972,0.880
8,WRENCH,12129,12615,0.971
9,<>,14353,14999,0.865


## 4. Side-by-Side: Raw vs Fully Processed

In [8]:
# Apply all rules in sequence
df_processed = df_raw.copy()
for _, rule in ALL_RULES:
    df_processed = rule.apply(df_processed, context)

comparison = df_raw.copy()
comparison.columns = [f'{c}_raw' for c in comparison.columns]

# If row counts differ, just show both
if len(comparison) == len(df_processed):
    for c in df_processed.columns:
        comparison[f'{c}_processed'] = df_processed[c].values
    display(comparison)
else:
    print('--- RAW ---')
    display(df_raw)
    print('\n--- PROCESSED ---')
    display(df_processed)

['DRYER', 'MACHINE']
['DRYER', 'MACHINE']
--- RAW ---


,Word,Onset,Offset,Probability
0,DRYER,335,1517,0.726
1,MACHINE,1577,2118,0.960
2,HELMET,2739,3280,0.962
3,PICKLE,4963,5463,0.907
4,EGGS,6345,8148,0.942
5,BAGELS,8649,9330,0.873
6,OREGANO,10251,10972,0.830
7,WRENCH,12134,12615,0.921
8,FISHBOWL,14358,14999,0.865
9,MOUTHWASH,16061,16842,0.882



--- PROCESSED ---


,Word,Onset,Offset,Probability
0,DRYER MACHINE,330,2118,0.776
1,<>,1330,2118,0.776
2,HELMET,2734,3280,1.000
3,<>,4958,5463,0.907
4,EGGS,6340,8148,0.992
5,<>,7340,8148,0.992
6,BAGELS,8644,9330,0.923
7,OREGANO,10246,10972,0.880
8,WRENCH,12129,12615,0.971
9,<>,14353,14999,0.865


## 5. Test Individual Rules in Isolation
Use this cell to test a single rule on custom data.

In [9]:
# Example: test PluralNormalization with synthetic data
test_df = pd.DataFrame({
    'Word': ['DOGS', 'TELEPHONE', 'FOXES', 'PARKED', 'UNKNOWN', 'LEADER'],
    'Onset': [100, 500, 1000, 1500, 2000, 2500],
    'Offset': [400, 900, 1400, 1900, 2400, 2900],
    'Probability': [0.9, 0.8, 0.7, 0.85, 0.6, 0.95],
})

rule = PluralNormalization()
result = rule.apply(test_df, context)

compare = test_df[['Word']].copy()
compare['After'] = result['Word']
compare.columns = ['Before', 'After']
compare['Changed'] = compare['Before'] != compare['After']
compare

,Before,After,Changed
0,DOGS,DOGS,False
1,TELEPHONE,TELEPHONE,False
2,FOXES,FOXES,False
3,PARKED,PARKED,False
4,UNKNOWN,UNKNOWN,False
5,LEADER,LEADER,False


In [10]:
# Example: test LongDurationVocalization with a word > 1s
test_df2 = pd.DataFrame({
    'Word': ['APPLE', 'UHHHHH', 'BANANA'],
    'Onset': [100, 1000, 5000],
    'Offset': [500, 3500, 5400],  # UHHHHH is 2500ms long
    'Probability': [0.9, 0.3, 0.8],
})

rule = LongDurationVocalization()
result = rule.apply(test_df2, context)
result

,Word,Onset,Offset,Probability
0,APPLE,100,500,0.9
1,UHHHHH,1000,3500,0.3
2,<>,2000,3000,0.3
3,<>,3000,3500,0.3
4,BANANA,5000,5400,0.8


## 6. Batch Test: All Trials in a Session

In [11]:
import glob

csv_dir = f'/home1/zrentala/automated_annotation/results/{TAG}/{SPLIT}{session_dir}/whisperx_out'
csv_files = sorted(glob.glob(os.path.join(csv_dir, '*.csv')),
                   key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))

summary = []
for csv_file in csv_files:
    trial_num = int(os.path.splitext(os.path.basename(csv_file))[0])
    df = pd.read_csv(csv_file)
    
    # Load trial-specific lst
    trial_lst = os.path.join(session_dir, f'{trial_num}.lst')
    if os.path.exists(trial_lst):
        with open(trial_lst) as f:
            context['file_list_words'] = {line.strip().upper() for line in f if line.strip()}
    else:
        context['file_list_words'] = None
    
    # Apply all rules
    df_proc = df.copy()
    for _, rule in ALL_RULES:
        df_proc = rule.apply(df_proc, context)
    
    wp = context.get('wordpool', set())
    lst = context.get('file_list_words', set()) or set()
    
    n_raw = len(df)
    n_proc = len(df_proc)
    n_vocalization = (df_proc['Word'] == '<>').sum() if len(df_proc) > 0 else 0
    n_in_lst = df_proc['Word'].apply(lambda w: w in lst).sum() if len(df_proc) > 0 else 0
    n_in_wp = df_proc['Word'].apply(lambda w: w in wp).sum() if len(df_proc) > 0 else 0
    
    summary.append({
        'Trial': trial_num,
        'Raw_Words': n_raw,
        'Processed_Words': n_proc,
        'In_LST': n_in_lst,
        'In_Wordpool': n_in_wp,
        'Vocalizations': n_vocalization,
    })

summary_df = pd.DataFrame(summary).set_index('Trial')
print(f'Session summary: {len(csv_files)} trials')
summary_df

KeyError: "None of ['Trial'] are in the columns"